# AuditFlow — Getting Started

This notebook walks you through the core AuditFlow features hands-on.
By the end you will have:

1. Verified that all services are running
2. Published an audit event and watched it flow through the pipeline
3. Explored the transformer and sink plugin registries
4. Experimented with conditional pipelines, data redaction, and idempotency

## Prerequisites

Start the local stack from the repo root:

```bash
just up          # builds the JAR, all images, starts the stack
just status      # wait until all containers show "healthy"
```


## 1. Setup

Configure service endpoints and helper functions used throughout the notebook.

In [1]:
import requests
import subprocess
import json
import time
import uuid
from datetime import datetime, timezone

# ── Service endpoints ─────────────────────────────────────────
BACKEND      = "http://localhost:8080/api/v1"
ACTUATOR     = "http://localhost:8080/actuator"
TRANSFORMER  = "http://localhost:8081"
SINK         = "http://localhost:8082"
RABBITMQ     = "http://localhost:15673/api"

# ── Docker container names (for log inspection) ──────────────
SINK_CONTAINER    = "auditflow-sink"
BACKEND_CONTAINER = "auditflow-backend"

# ── Timing ───────────────────────────────────────────────────
ASYNC_WAIT = 4   # seconds to wait for async pipeline processing
TIMEOUT    = 10  # HTTP request timeout

print("Configuration ready.")

Configuration ready.


In [2]:
def docker_logs(container: str, since: datetime) -> list[str]:
    """Fetch container stdout+stderr since the given UTC timestamp."""
    ts = since.strftime("%Y-%m-%dT%H:%M:%SZ")
    r = subprocess.run(
        ["docker", "logs", container, "--since", ts],
        capture_output=True, text=True
    )
    return (r.stdout + r.stderr).splitlines()


def show(resp, label=""):
    """Pretty-print an HTTP response."""
    prefix = f"{label}: " if label else ""
    print(f"{prefix}HTTP {resp.status_code}")
    try:
        print(json.dumps(resp.json(), indent=2))
    except Exception:
        print(resp.text[:500])


print("Helpers ready.")

Helpers ready.


## 2. Health Checks

Verify that the transformer and sink services are reachable.
The backend exposes health at `/actuator/health`; the Python services use `/health`.

In [3]:
for name, url in [
    ("backend",      f"{ACTUATOR}/health"),
    ("transformer",  f"{TRANSFORMER}/health"),
    ("sink",         f"{SINK}/health"),
]:
    try:
        r = requests.get(url, timeout=TIMEOUT)
        status = r.json().get("status", "UNKNOWN")
        print(f"  {name:15s} → {status}")
    except requests.ConnectionError:
        print(f"  {name:15s} → UNREACHABLE — is the stack running?")

  backend         → UP
  transformer     → UP
  sink            → UP


## 3. Plugin Registries

Each Python service dynamically loads transformer/sink modules.
The `/registry` endpoint lists what is available.

**Transformer modules** must expose `transform(input_data: dict) -> dict`.
**Sink modules** must expose `process(event_data: dict, properties: dict) -> dict`.

In [4]:
print("Available transformers:")
r = requests.get(f"{TRANSFORMER}/registry", timeout=TIMEOUT)
for t in r.json().get("transformers", []):
    print(f"  {t['id']:25s} v{t.get('version','?')}  {t.get('description','')}")

print("\nAvailable sinks:")
r = requests.get(f"{SINK}/registry", timeout=TIMEOUT)
for s in r.json().get("sinks", []):
    print(f"  {s['id']:25s} v{s.get('version','?')}  {s.get('description','')}")

Available transformers:
  audit_loki                v1.0.0  Loki transformer: reshapes an AuditFlow event into a Grafana Loki push payload.
  audit_opensearch          v1.0.0  OpenSearch transformer: flattens an AuditFlow event into an OpenSearch-friendly document.
  zero                      v1.0.0  Pass-through transformer: returns the input event unchanged.

Available sinks:
  aws_cloudwatch_sink       v1.0.0  AWS CloudWatch Logs Sink - Send events to AWS CloudWatch Logs.
  aws_s3_sink               v1.0.0  AWS S3 Sink - Store events in Amazon S3.
  azure_blob_sink           v1.0.0  Azure Blob Storage Sink - Store events in Azure Blob Storage.
  datadog_sink              v1.0.0  Datadog Sink - forward audit events to the Datadog Logs intake API.
  gcs_sink                  v1.0.0  Google Cloud Storage Sink - Store events in Google Cloud Storage.
  logging_sink              v1.0.0  Logging Sink - Simple sink that logs events.
  loki_sink                 v1.0.0  Loki Sink - Send event

## 4. Publish Your First Audit Event

The happy path: publish an event via REST, the backend buffers it in RabbitMQ,
the consumer picks it up, runs it through the `zero` transformer (pass-through),
and delivers it to `logging_sink` which prints it to stdout.

**Input** — the JSON payload we send:
- `eventId` — unique identifier (used for idempotency dedup)
- `eventType` — semantic label (e.g. `user.login`)
- `sourceSystem` — originating service
- `tenantId` — multi-tenant isolation key
- `extra` — arbitrary metadata bag

**Output** — the backend returns `200 OK` with a confirmation body.
After a few seconds the sink container log will contain the processed event.

In [5]:
event_id = str(uuid.uuid4())
ts_before = datetime.now(timezone.utc)

event = {
    "eventId":      event_id,
    "eventType":    "user.login",
    "sourceSystem": "auth-service",
    "tenantId":     "T-DEMO-001",
    "extra": {
        "userId":    "alice",
        "sessionId": "sess-abc-123",
        "action_status": "SUCCESS"
    }
}

print("Sending event:")
print(json.dumps(event, indent=2))
print()

resp = requests.post(f"{BACKEND}/audit/publish", json=event, timeout=TIMEOUT)
show(resp, "Response")

Sending event:
{
  "eventId": "0873bf5b-6286-4ece-b773-00a3dcba126f",
  "eventType": "user.login",
  "sourceSystem": "auth-service",
  "tenantId": "T-DEMO-001",
  "extra": {
    "userId": "alice",
    "sessionId": "sess-abc-123",
    "action_status": "SUCCESS"
  }
}

Response: HTTP 200
Audit event published successfully


### Verify Sink Delivery

After the async pipeline runs, the sink container log should contain our event.
We search for the `eventId` in recent log lines.

In [6]:
time.sleep(ASYNC_WAIT)

lines = docker_logs(SINK_CONTAINER, ts_before)
matches = [l for l in lines if event_id in l]

print(f"Sink log lines containing event {event_id[:8]}...: {len(matches)}")
for l in matches[:3]:
    print(f"  {l}")

if matches:
    print("\nEvent was delivered to the sink successfully.")
else:
    print("\nNo matching lines found. Check: just log sink")

Sink log lines containing event 0873bf5b...: 2
  2026-06-26 15:16:18,199 - INFO - Processing event through sink 'logging_sink'. Raw body: {'event_data': {'timestamp': '2026-06-26T15:16:18.17163767Z', 'eventId': '0873bf5b-6286-4ece-b773-00a3dcba126f', 'eventTime': None, 'correlationId': '166daf12-153d-40e2-8392-7a91fd726504', 'eventType': 'user.login', 'sourceSystem': 'auth-service', 'tenantId': 'T-DEMO-001', 'geolocation': None, 'extra': {'userId': '***', 'action_status': 'SUCCESS'}}, 'properties': {'format': 'json', 'log-level': 'DEBUG'}}
    "eventId": "0873bf5b-6286-4ece-b773-00a3dcba126f",

Event was delivered to the sink successfully.


## 5. Idempotency (Duplicate Suppression)

Publishing the same `eventId` twice should result in exactly one sink delivery.
The API always returns `200` (it accepts the message), but the consumer deduplicates before processing.

The default stack uses an in-memory dedup store.

In [7]:
dedup_id = str(uuid.uuid4())
ts_dedup = datetime.now(timezone.utc)

dedup_event = {
    "eventId":      dedup_id,
    "eventType":    "order.created",
    "sourceSystem": "order-service",
    "tenantId":     "T-DEMO-004"
}

print("First publish:")
resp1 = requests.post(f"{BACKEND}/audit/publish", json=dedup_event, timeout=TIMEOUT)
show(resp1, "Response")
time.sleep(ASYNC_WAIT)

print("\nDuplicate publish (same eventId):")
resp2 = requests.post(f"{BACKEND}/audit/publish", json=dedup_event, timeout=TIMEOUT)
show(resp2, "Response")

First publish:
Response: HTTP 200
Audit event published successfully

Duplicate publish (same eventId):
Response: HTTP 200
Audit event published successfully


### Verify Dedup

The backend log should show a dedup rejection for the second publish.
The sink log should show exactly one delivery.

In [8]:
time.sleep(ASYNC_WAIT)

# Check backend for dedup log
be_lines = docker_logs(BACKEND_CONTAINER, ts_dedup)
dedup_logs = [
    l for l in be_lines
    if dedup_id in l
    and any(kw in l.lower() for kw in ("duplicate", "already", "idempoten", "dedup"))
]
print(f"Backend dedup log lines: {len(dedup_logs)}")
for l in dedup_logs[:2]:
    print(f"  {l}")

# Check sink for single delivery
sink_lines = docker_logs(SINK_CONTAINER, ts_dedup)
deliveries = [l for l in sink_lines if "order.created" in l and "T-DEMO-004" in l]
print(f"\nSink deliveries: {len(deliveries)}")
for l in deliveries:
    print(f"  {l}")

if len(deliveries) == 1:
    print("\nIdempotency working: exactly one delivery for duplicate events.")
elif len(deliveries) == 0:
    print("\nNo deliveries found. Check: just log sink")
else:
    print(f"\nExpected 1 delivery, got {len(deliveries)}. Dedup may not be active.")

Backend dedup log lines: 2
  {"timestamp":"2026-06-26 15:16:26.294+0000","app":"auditflow-backend","pod":"886f93dab02e","correlationId":"","traceId":"49332f544f1197b0f35dae40f5c56dde","spanId":"07188435d6e50601","thread":"labs64-audit-topic.labs64.io-auditflow-2","level":"DEBUG","logger":"io.labs64.audit.service.InMemoryIdempotencyService","message":"Duplicate or in-flight eventId '9782ba54-a92b-4b02-b13a-a27a4b0d27c8', claim refused"}
  {"timestamp":"2026-06-26 15:16:26.295+0000","app":"auditflow-backend","pod":"886f93dab02e","correlationId":"","traceId":"49332f544f1197b0f35dae40f5c56dde","spanId":"07188435d6e50601","thread":"labs64-audit-topic.labs64.io-auditflow-2","level":"DEBUG","logger":"io.labs64.audit.service.AuditService","message":"Duplicate audit event eventId='9782ba54-a92b-4b02-b13a-a27a4b0d27c8', dropping."}

Sink deliveries: 1
  2026-06-26 15:16:22,273 - INFO - Processing event through sink 'logging_sink'. Raw body: {'event_data': {'timestamp': '2026-06-26T15:16:22.25615

## 6. Exploring the System

A few useful endpoints for understanding what the pipeline is doing.

In [9]:
# Consumer metrics (how many events have been processed)
print("Consumer metrics:")
try:
    r = requests.get(f"{ACTUATOR}/metrics/auditflow.consumer.events.processed", timeout=TIMEOUT)
    value = r.json()["measurements"][0]["value"]
    print(f"  events.processed = {value}")
except Exception as e:
    print(f"  Could not read metric: {e}")

Consumer metrics:
  events.processed = 16.0


In [10]:
# RabbitMQ queue depth (should be 0 after all events are consumed)
print("RabbitMQ queue depth:")
try:
    queue = "labs64-audit-topic.labs64.io-auditflow"
    r = requests.get(
        f"{RABBITMQ}/queues/%2F/{queue}",
        auth=("guest", "guest"),
        timeout=TIMEOUT
    )
    ready = r.json()["messages_ready"]
    print(f"  messages_ready = {ready}")
except Exception as e:
    print(f"  Could not check queue: {e}")

RabbitMQ queue depth:
  messages_ready = 0


## Next Steps

- **Add a custom transformer**: drop a Python file into `auditflow-transformer/transformers/`
  implementing `def transform(input_data: dict) -> dict`.
- **Add a custom sink**: same pattern in `auditflow-sink/sinks/` with
  `def process(event_data: dict, properties: dict) -> dict`.
- **Configure pipelines**: edit `auditflow.pipelines` in `application.yml` or
  override via `JAVA_OPTS` in docker-compose.
- **Enable observability**: run `just up obs` to add Grafana, Prometheus, Tempo, and Loki.
- **Read the API spec**: open `http://localhost:8080/swagger-ui.html` when the stack is up.